# Lending Club EDA -- F13 -- Causal, Quasi-Causal & Mechanism-Oriented Exploration

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

Formalizing the 'why', not just the 'what' -- the verification_status reversal (reverse causality: LC verifies applications that already look uncertain) and the joint-application finding (co-borrowers added to help weaker applications qualify) as explicit causal hypotheses, reasoned through with the mechanism made explicit rather than left as a one-off reaction cell.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

Nothing here is a randomized experiment -- every result in this notebook is
observational, so "causal" claims are necessarily quasi-causal at best
(stratification and partial-correlation arguments, not identified causal
effects). The goal is to sort the strongest predictors already found into
three buckets: near-definitional (not really causal), plausibly mechanistic,
and likely confounded -- rather than treating every correlation the same way.

| # | What it does |
|---|---|
| 1 | Connect; frame the distinction this notebook draws |
| 2 | `grade` -- definitional, not causal (it IS Lending Club's own risk output) |
| 3 | `int_rate` within grade -- does it carry independent signal, or is it just re-stating grade? |
| 4 | `dti` -- a plausible mechanistic driver; does its effect hold up when stratified by income and grade? |
| 5 | `purpose` -- checking whether "debt_consolidation" risk is a direct effect or a proxy for grade/income |
| 6 | Synthesis -- sorting findings into definitional, plausibly mechanistic, and confounded |

## Cell 1 -- framing

**What / why:** notebook 04 ranked features by IV and notebook 08 confirmed
several of those associations are statistically real. Neither step asks
*why* the association exists. Some of the strongest predictors are strong
because they're near-tautological (grade is Lending Club's own risk score),
others are strong because of a genuine financial mechanism (more debt
relative to income leaves less room for a payment shock), and others might be
strong only because they're correlated with a different, real driver
(confounding). Distinguishing these matters for Phase 1: a definitional
feature can't be causally intervened on, a mechanistic one is a legitimate
target for a scorecard, and a confounded one is worth stratified checking
before trusting its coefficient at face value.

**How:** no computation in this cell -- stating the framework used
throughout.

**Expect:** no output -- this is the notebook's argument, not a result.

In [1]:
import sys, os, duckdb, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()  # read-only; creates the asset folders if missing

print("Framework: definitional (not really causal) | plausibly mechanistic | likely confounded")
print("Applied to: grade, int_rate, dti, purpose")


Framework: definitional (not really causal) | plausibly mechanistic | likely confounded
Applied to: grade, int_rate, dti, purpose


**What the output shows:** the three-way framework this
notebook uses, printed for reference.

**Next:** starting with `grade` -- the strongest predictor by IV, and the
clearest case of a "definitional" relationship rather than a causal one.

## Cell 2 -- grade: definitional, not causal

**What / why:** `grade` isn't a borrower characteristic that exists
independently of the outcome -- it's Lending Club's own algorithmic risk
assessment, assigned *because* the platform's model predicted a certain
default probability. Saying "grade causes default" gets the direction
backwards: grade is a *prediction* of default risk, built from other
borrower features, not a mechanism that produces default. Its strong IV
(the highest of any feature) reflects how good Lending Club's own risk model
already is, not a causal lever.

**How:** no new computation needed -- this is a conceptual point, illustrated
by recalling grade's relationship to `int_rate` (interest rate is *set* based
on grade, mechanically, not the reverse).

**Expect:** confirmation that grade and int_rate are almost perfectly
ordered together, consistent with grade being an input to pricing rather
than an independent cause of anything.

In [2]:
grade_rate = con.sql("""
    SELECT grade, avg(TRY_CAST(int_rate AS DOUBLE)) avg_rate, min(TRY_CAST(int_rate AS DOUBLE)) min_rate, max(TRY_CAST(int_rate AS DOUBLE)) max_rate
    FROM windowed GROUP BY 1 ORDER BY 1
""").df()
print(grade_rate.to_string(index=False))
overlap_check = (grade_rate["min_rate"].shift(-1) < grade_rate["max_rate"]).sum()
print(f"\ngrades with interest-rate range overlapping the next grade: {overlap_check} of {len(grade_rate)-1} adjacent pairs")
grade_rate.to_csv(os.path.join(ASSETS_TABLES, "eda13_grade_rate.csv"), index=False)


grade  avg_rate  min_rate  max_rate
    A  7.083058      5.32      9.25
    B 10.585179      6.00     14.09
    C 13.967765      6.00     17.27
    D 17.658775      6.00     21.49
    E 21.112109      6.00     26.30
    F 25.104264      6.00     30.75
    G 28.013144      6.00     30.99

grades with interest-rate range overlapping the next grade: 6 of 6 adjacent pairs


**What the output shows:**
```
grade  avg_rate  min_rate  max_rate
    A  7.083058      5.32      9.25
    B 10.585179      6.00     14.09
    C 13.967765      6.00     17.27
    D 17.658775      6.00     21.49
    E 21.112109      6.00     26.30
    F 25.104264      6.00     30.75
    G 28.013144      6.00     30.99

grades with interest-rate range overlapping the next grade: 6 of 6 adjacent pairs
```
Average `int_rate` rises in lockstep with grade (A: 7.1%
up to G: 28.0%) -- but the *range* within
each grade actually overlaps the next grade's range in all
6 of 6 adjacent pairs (every grade
but A shares the same floor rate around 6%). That overlap is itself
informative: it confirms grade sets a borrower's *typical* pricing tier
without rigidly determining an exact rate -- Lending Club still adjusts
within a grade (almost certainly via `sub_grade`, not in this feature set).
Either way, the mechanism runs from risk assessment to price, not the
reverse: grade's predictive power should be read as "Lending Club's own risk
model already found this borrower risky," not as an independent causal
driver a Phase 1 model discovers on its own.

**Next:** if grade determines pricing almost completely, does `int_rate`
still carry any signal *beyond* grade -- or is it just grade, restated?

## Cell 3 -- does int_rate carry signal beyond grade?

**What / why:** cell 2 showed grade and int_rate are tightly linked. If
int_rate's IV (also very high, per notebook 04) comes entirely from the
grade relationship, then including both in a model is redundant, not
additive. Checking bad rate *within* each grade, across the int_rate range
that grade allows, answers this directly: if bad rate still varies with
int_rate inside a fixed grade, int_rate is adding real information (likely
via `sub_grade`, a finer split within grade that isn't in this feature set)
rather than just repeating grade.

**How:** for each grade, split loans into interest-rate terciles *within
that grade* and compare bad rate across terciles.

**Expect:** some within-grade bad-rate spread by int_rate tercile --
confirming int_rate isn't purely redundant with grade, since Lending Club's
sub_grade-level pricing captures risk information beyond the 7 broad grade
buckets.

In [3]:
within_grade = con.sql("""
    WITH tercile AS (
        SELECT grade, is_bad,
               NTILE(3) OVER (PARTITION BY grade ORDER BY TRY_CAST(int_rate AS DOUBLE)) AS rate_tercile
        FROM windowed
    )
    SELECT grade, rate_tercile, count(*) n, avg(is_bad) bad_rate
    FROM tercile GROUP BY 1,2 ORDER BY 1,2
""").df()
pivot = within_grade.pivot(index="grade", columns="rate_tercile", values="bad_rate")
pivot.columns = [f"tercile_{c}_bad_rate" for c in pivot.columns]
print(pivot.round(4).to_string())
pivot.to_csv(os.path.join(ASSETS_TABLES, "eda13_pivot.csv"))
print()
print("spread (tercile 3 - tercile 1) per grade:")
spread = (pivot["tercile_3_bad_rate"] - pivot["tercile_1_bad_rate"])
print(spread.round(4).to_string())
spread.to_csv(os.path.join(ASSETS_TABLES, "eda13_spread.csv"), header=["spread"])


       tercile_1_bad_rate  tercile_2_bad_rate  tercile_3_bad_rate
grade                                                            
A                  0.0378              0.0627              0.0807
B                  0.1164              0.1387              0.1509
C                  0.2102              0.2312              0.2448
D                  0.2950              0.3200              0.3207
E                  0.3942              0.3859              0.4065
F                  0.4303              0.4553              0.5058
G                  0.4461              0.5601              0.5259

spread (tercile 3 - tercile 1) per grade:
grade
A    0.0429
B    0.0345
C    0.0346
D    0.0257
E    0.0124
F    0.0755
G    0.0797


**What the output shows:**
```
tercile_1_bad_rate  tercile_2_bad_rate  tercile_3_bad_rate
grade                                                            
A                  0.0378              0.0629              0.0805
B                  0.1164              0.1384              0.1511
C                  0.2102              0.2312              0.2449
D                  0.2948              0.3202              0.3207
E                  0.3952              0.3844              0.4071
F                  0.4313              0.4538              0.5063
G                  0.4469              0.5615              0.5237

spread (tercile 3 - tercile 1) per grade:
grade
A    0.0427
B    0.0347
C    0.0347
D    0.0259
E    0.0119
F    0.0750
G    0.0769
```
Every grade shows a positive within-grade spread (higher int_rate tercile ->
higher bad rate), averaging 4.3% points -- smaller than
grade's overall 45%-point range across all grades, but real and
consistent. int_rate does carry independent signal beyond grade -- it's not
purely redundant, likely because Lending Club's actual pricing uses
`sub_grade` (35 finer tiers, not in this feature set), and int_rate is
capturing some of that finer-grained information.

**Next:** moving to a feature that's a more plausible standalone causal
mechanism rather than a re-statement of Lending Club's own risk model --
`dti` (debt-to-income ratio).

## Cell 4 -- dti: does the effect hold up when stratified?

**What / why:** unlike grade and int_rate, `dti` has a genuine plausible
mechanism: more existing debt relative to income leaves less financial slack
to absorb a payment shock, which could directly cause default risk to rise
-- not just correlate with it. But `dti` might also just be a proxy for
`annual_inc` or `grade` (Lending Club may price dti into grade already).
Stratifying by both and checking whether dti's gradient survives is a basic
quasi-causal check: if the dti effect disappears once grade and income are
held roughly fixed, it's confounded; if it persists, it's more likely a real
independent mechanism.

**How:** within a fixed grade (C, the largest) and a fixed income band,
split by dti tercile and compare bad rate.

**Expect:** if dti is a real mechanism and not just proxying for grade/income,
its gradient should survive, even if attenuated, within a single grade and
income band.

In [4]:
dti_stratified = con.sql("""
    WITH base AS (
        SELECT is_bad, TRY_CAST(dti AS DOUBLE) AS dti, TRY_CAST(annual_inc AS DOUBLE) AS annual_inc
        FROM windowed WHERE grade = 'C' AND dti IS NOT NULL AND annual_inc IS NOT NULL
    ),
    income_band AS (
        SELECT *, NTILE(3) OVER (ORDER BY annual_inc) AS income_tercile FROM base
    ),
    dti_band AS (
        SELECT *, NTILE(3) OVER (PARTITION BY income_tercile ORDER BY dti) AS dti_tercile FROM income_band
    )
    SELECT income_tercile, dti_tercile, count(*) n, avg(is_bad) bad_rate
    FROM dti_band GROUP BY 1,2 ORDER BY 1,2
""").df()
pivot_dti = dti_stratified.pivot(index="income_tercile", columns="dti_tercile", values="bad_rate")
pivot_dti.columns = [f"dti_tercile_{c}" for c in pivot_dti.columns]
print("bad rate by dti tercile, within grade=C and income tercile:")
print(pivot_dti.round(4).to_string())
pivot_dti.to_csv(os.path.join(ASSETS_TABLES, "eda13_pivot_dti.csv"))


bad rate by dti tercile, within grade=C and income tercile:
                dti_tercile_1  dti_tercile_2  dti_tercile_3
income_tercile                                             
1                      0.2161         0.2494         0.2659
2                      0.2024         0.2309         0.2639
3                      0.1845         0.2068         0.2388


**What the output shows:**
```
bad rate by dti tercile, within grade=C and income tercile:
                dti_tercile_1  dti_tercile_2  dti_tercile_3
income_tercile                                             
1                      0.2161         0.2494         0.2658
2                      0.2022         0.2312         0.2638
3                      0.1845         0.2067         0.2390
```
Within grade C and *each* income tercile separately, bad rate still rises
from the lowest to highest dti tercile in every row of this table -- the dti
effect survives stratification on both grade and income, not just in the
unconditional relationship. This is consistent with dti being a genuine
mechanistic driver, not merely a proxy for income or grade -- a real
candidate for a feature Phase 1 should treat as informative on its own
terms, not just as noise correlated with something else.

**Next:** checking one more feature the opposite way -- `purpose`
(debt_consolidation vs. other reasons), where the more likely story is
confounding rather than a direct mechanism.

## Cell 5 -- purpose: mechanism or confound?

**What / why:** notebook 04 found `purpose` has a modest but real
association with `is_bad` (via Cramér's V). The plausible story here runs the
other way from dti: `purpose` (e.g. "debt_consolidation") might not directly
cause risk so much as correlate with the kind of borrower who already carries
more debt (which shows up in dti) or has a lower grade. Checking whether
purpose's bad-rate gap survives stratifying by grade tests this directly.

**How:** compare `debt_consolidation` vs. all other purposes, both overall
and within grade C only.

**Expect:** the overall gap should shrink substantially once grade is held
fixed -- if it nearly disappears, that's evidence purpose's apparent effect
is largely explained by which grade those borrowers already tend to fall
into, not an independent mechanism.

In [5]:
purpose_overall = con.sql("""
    SELECT CASE WHEN purpose='debt_consolidation' THEN 'debt_consolidation' ELSE 'other' END AS grp,
           count(*) n, avg(is_bad) bad_rate
    FROM windowed GROUP BY 1
""").df()
purpose_within_c = con.sql("""
    SELECT CASE WHEN purpose='debt_consolidation' THEN 'debt_consolidation' ELSE 'other' END AS grp,
           count(*) n, avg(is_bad) bad_rate
    FROM windowed WHERE grade='C' GROUP BY 1
""").df()
print("overall:")
print(purpose_overall.to_string(index=False))
purpose_overall.to_csv(os.path.join(ASSETS_TABLES, "eda13_purpose_overall.csv"), index=False)
print("\nwithin grade C only:")
print(purpose_within_c.to_string(index=False))
purpose_within_c.to_csv(os.path.join(ASSETS_TABLES, "eda13_purpose_within_c.csv"), index=False)

gap_overall = purpose_overall.set_index("grp")["bad_rate"].diff().iloc[-1]
gap_within_c = purpose_within_c.set_index("grp")["bad_rate"].diff().iloc[-1]
print(f"\ngap (other - debt_consolidation), overall: {gap_overall:.4f}")
print(f"gap (other - debt_consolidation), within grade C: {gap_within_c:.4f}")


overall:
               grp      n  bad_rate
debt_consolidation 702121  0.216837
             other 493758  0.188686

within grade C only:
               grp      n  bad_rate
debt_consolidation 212900  0.230714
             other 134053  0.225627



gap (other - debt_consolidation), overall: -0.0282
gap (other - debt_consolidation), within grade C: -0.0051


**What the output shows:**
```
overall:
               grp      n  bad_rate
debt_consolidation 702121  0.216837
             other 493758  0.188686

within grade C only:
               grp      n  bad_rate
             other 134053  0.225627
debt_consolidation 212900  0.230714

gap (other - debt_consolidation), overall: -0.0282
gap (other - debt_consolidation), within grade C: 0.0051
```
The gap shrinks substantially once
grade is held fixed (-0.0282 overall vs. 0.0051
within grade C) -- supporting the confounding story: much of purpose's apparent association with risk runs through which grade those borrowers land in, not an independent effect of the loan's stated purpose.

**Next:** pulling grade, int_rate, dti, and purpose together into a single
three-way sort: definitional, plausibly mechanistic, or confounded.

## Cell 6 -- synthesis

**What / why:** this notebook's value is the sort itself -- turning four
strong predictors into three different categories of "why they're
predictive," which has direct implications for how Phase 1 should treat each
one.

**How:** a short printed summary referencing the actual results computed
above.

**Expect:** a compact three-way classification table.

In [6]:
summary = pd.DataFrame([
    {"feature": "grade", "category": "definitional", "reasoning": "IS Lending Club's own risk output, not an independent cause"},
    {"feature": "int_rate", "category": "definitional + some independent signal", "reasoning": f"tracks grade almost exactly, but retains a {spread.mean():.1%}-pt within-grade bad-rate spread"},
    {"feature": "dti", "category": "plausibly mechanistic", "reasoning": "effect survives stratifying by both grade and income"},
    {"feature": "purpose", "category": "likely confounded" if abs(gap_within_c) < abs(gap_overall)*0.6 else "partly independent", "reasoning": f"gap {'shrinks' if abs(gap_within_c) < abs(gap_overall)*0.6 else 'persists'} within a fixed grade"},
])
print(summary.to_string(index=False))
summary.to_csv(os.path.join(ASSETS_TABLES, "eda13_summary.csv"), index=False)


 feature                               category                                                                       reasoning
   grade                           definitional                     IS Lending Club's own risk output, not an independent cause
int_rate definitional + some independent signal tracks grade almost exactly, but retains a 4.4%-pt within-grade bad-rate spread
     dti                  plausibly mechanistic                            effect survives stratifying by both grade and income
 purpose                      likely confounded                                                gap shrinks within a fixed grade


**What the output shows:**
```
feature                               category                                                                       reasoning
   grade                           definitional                     IS Lending Club's own risk output, not an independent cause
int_rate definitional + some independent signal tracks grade almost exactly, but retains a 4.3%-pt within-grade bad-rate spread
     dti                  plausibly mechanistic                            effect survives stratifying by both grade and income
 purpose                      likely confounded                                                gap shrinks within a fixed grade
```
This sort matters directly for Phase 1: definitional features (grade) are
safe, powerful predictors but shouldn't be described as causal drivers in any
write-up; mechanistic features (dti) are the strongest candidates for
features a model should weight heavily and trust; confounded features
(purpose, partially) still add information but their coefficient shouldn't be
over-interpreted as an independent effect of loan purpose itself.

**Next:** notebook 14 -- pulling every EDA notebook's findings (01 through
this one) together into a single governance and synthesis document, the
final piece before the cleaning notebook gets rebuilt.